# Kiểm tra Intent Classifier và Multi-Agent Workflow

Notebook này thực hiện ba bước tuần tự:

1. Kiểm tra logic phân loại intent cho câu hỏi multi-agent.
2. Xác nhận bộ định tuyến (`Router`) kích hoạt đúng các agent phối hợp.
3. Chạy thử toàn bộ luồng để kiểm tra phản hồi cuối cùng.


In [1]:
from pathlib import Path
import sys

# ensure we can import from the workspace root
workspace_root = Path.cwd()
sys.path.insert(0, str(workspace_root))

from multi_agent_chatbot_v3 import IntentClassifierAgent, RouterAgent

# Step 1: Quick intent classifier check
intent_agent = IntentClassifierAgent()
query = 'cậu có phải là multi-agent không?'
intent = intent_agent._quick_intent(query)
print('Step 1: detected intent =', intent)


C:\Users\DUC\AppData\Roaming\Python\Python312\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Step 1: detected intent = langgraph


In [2]:
# Step 2: Verify router dispatch logic for multi-agent intent
router_agent = RouterAgent()
route_result = router_agent.run(query=query, intent=intent, kb_context='')
print('\nStep 2: router dispatched agents =', [t['agent'] for t in route_result.get('_sub_timings', [])])
print('Workspace messages:', route_result.get('workspace_messages'))
print('Result keys:', [k for k in route_result.keys() if k not in ['workspace_messages', '_sub_timings']])



Step 2: router dispatched agents = ['WebSearch', 'KnowledgeBase', 'Analyst', 'Planner', 'Validator']
Workspace messages: ["KnowledgeBase đã cung cấp ngữ cảnh cho intent 'langgraph'.", "Analyst đã phân tích chủ đề 'cậu có phải là multi-agent không?' và cung cấp luận giải chuyên sâu.", 'Planner đã xác định cấu trúc nhiệm vụ và cách các agent hợp tác.', 'Validator đã xác nhận tính khả thi và gợi ý phối hợp giữa các agent.']
Result keys: ['web_results', 'kb_context', 'analysis', 'planning', 'validation']


In [3]:
# Step 3: Run the full graph and inspect the intent and final answer keys
from multi_agent_chatbot_v3 import build_graph

graph = build_graph()
initial_state = {
    'session_id': 'test',
    'turn_id': 1,
    'user_query': query,
    'user_id': 'test',
    'intent': '',
    'goal': '',
    'kb_context': '',
    'workspace_messages': [],
    'coordination_summary': '',
    'planning': '',
    'validation': '',
    'web_results': '',
    'calc_result': '',
    'datetime_result': '',
    'analysis': '',
    'final_answer': '',
    'conversation_history': [],
    'agent_timings': [],
    'call_graph': [],
    'log': [],
}

result = graph.invoke(initial_state)
print('\nStep 3: full graph result keys:', sorted(result.keys()))
print('Graph intent:', result.get('intent'))
print('Coordination summary:\n', result.get('coordination_summary'))
print('Final answer preview:\n', result.get('final_answer')[:400] if result.get('final_answer') else 'N/A')


Groq rate limit detected (tokens_per_day) for model=compound-beta. attempt=0. error=error code: 429 - {'error': 
{'message': 'rate limit reached for model `openai/gpt-oss-120b` in organization `org_01ksthcq6re5m93yqpcjm1eq4h` 
service tier `on_demand` on tokens per day (tpd): limit 200000, used 198727, requested 4187. please try again in 
20m58.848s. need more tokens? upgrade to dev tier today at https://console.groq.com/settings/billing', 'type': 
'compound', 'code': 'rate_limit_exceeded'}}

Groq rate limit detected (tokens_per_day) for model=compound-beta. attempt=0. error=error code: 429 - {'error': 
{'message': 'rate limit reached for model `openai/gpt-oss-120b` in organization `org_01ksthcq6re5m93yqpcjm1eq4h` 
service tier `on_demand` on tokens per day (tpd): limit 200000, used 198704, requested 8604. please try again in 
52m37.056s. need more tokens? upgrade to dev tier today at https://console.groq.com/settings/billing', 'type': 
'compound', 'code': 'rate_limit_exceeded'}}

GroqQuotaExceededError: Groq quota exceeded (tokens_per_day): error code: 429 - {'error': {'message': 'rate limit reached for model `openai/gpt-oss-120b` in organization `org_01ksthcq6re5m93yqpcjm1eq4h` service tier `on_demand` on tokens per day (tpd): limit 200000, used 198704, requested 8604. please try again in 52m37.056s. need more tokens? upgrade to dev tier today at https://console.groq.com/settings/billing', 'type': 'compound', 'code': 'rate_limit_exceeded'}}